# 🏗️ Lakehouse Playground — Setup & Hello World

This notebook initializes the Lakehouse environment and runs your first queries.

Since this notebook runs **inside Docker Compose**, all services are reachable by hostname.

---
## ⚙️ Step 1 — Configuration

In [1]:
import boto3
import requests
import json
import time
from botocore.client import Config

# --- S3 (SeaweedFS) ---
S3_ENDPOINT = "http://seaweedfs:8333"
S3_ACCESS_KEY = "lakehouse-admin"
S3_SECRET_KEY = "lakehouse-secret-key"
S3_REGION = "us-east-1"
S3_BUCKET = "lakehouse"

# --- Polaris (both REST and Management APIs are on port 8181) ---
POLARIS_URL = "http://polaris:8181"
POLARIS_CLIENT_ID = "root"
POLARIS_CLIENT_SECRET = "polaris-secret"

# --- Trino ---
TRINO_HOST = "trino"
TRINO_PORT = 8080

# --- Medallion layers ---
NAMESPACES = ["bronze", "silver", "gold"]

print("✅ Configuration loaded")

✅ Configuration loaded


---
## 🔍 Step 2 — Wait for Services

In [ ]:
def wait_for_service(name, url, max_attempts=30):
    """Wait for a service to be ready."""
    print(f"⏳ Waiting for {name}...", end="")
    for attempt in range(max_attempts):
        try:
            r = requests.get(url, timeout=2)
            if r.ok:
                print(f" ✅ {name} is ready!")
                return True
        except requests.ConnectionError:
            pass
        print(".", end="", flush=True)
        time.sleep(2)
    print(f" ❌ {name} did not become ready.")
    return False

wait_for_service("SeaweedFS S3", S3_ENDPOINT)
wait_for_service("Polaris REST API", f"{POLARIS_URL}/api/catalog/v1/config")
wait_for_service("Trino", f"http://{TRINO_HOST}:{TRINO_PORT}/v1/info")

---
## 🪣 Step 3 — Create S3 Bucket

In [2]:
s3 = boto3.client(
    "s3",
    endpoint_url=S3_ENDPOINT,
    aws_access_key_id=S3_ACCESS_KEY,
    aws_secret_access_key=S3_SECRET_KEY,
    region_name=S3_REGION,
    config=Config(signature_version="s3v4"),
)

existing = [b["Name"] for b in s3.list_buckets().get("Buckets", [])]

if S3_BUCKET in existing:
    print(f"⚠️  Bucket '{S3_BUCKET}' already exists — skipping")
else:
    s3.create_bucket(Bucket=S3_BUCKET)
    print(f"✅ Created bucket: {S3_BUCKET}")

print(f"\n📋 Bucket ready: 🪣 {S3_BUCKET}")
for ns in NAMESPACES:
    print(f"   └── s3://{S3_BUCKET}/{ns}/")

✅ Created bucket: lakehouse

📋 Bucket ready: 🪣 lakehouse
   └── s3://lakehouse/bronze/
   └── s3://lakehouse/silver/
   └── s3://lakehouse/gold/


---
## 🔑 Step 4 — Get Polaris API Token

In [3]:
token_resp = requests.post(
    f"{POLARIS_URL}/api/catalog/v1/oauth/tokens",
    data={
        "grant_type": "client_credentials",
        "client_id": POLARIS_CLIENT_ID,
        "client_secret": POLARIS_CLIENT_SECRET,
        "scope": "PRINCIPAL_ROLE:ALL",
    },
)

if token_resp.ok:
    POLARIS_TOKEN = token_resp.json()["access_token"]
    headers = {"Authorization": f"Bearer {POLARIS_TOKEN}", "Content-Type": "application/json"}
    print(f"✅ Polaris token obtained (expires in {token_resp.json().get('expires_in', '?')}s)")
else:
    print(f"❌ Failed to get token: {token_resp.status_code} — {token_resp.text}")

✅ Polaris token obtained (expires in 3600s)


---
## 📚 Step 5 — Register Polaris Catalog

In [4]:
catalog_payload = {
    "catalog": {
        "name": "lakehouse",
        "type": "INTERNAL",
        "properties": {
            "default-base-location": f"s3://{S3_BUCKET}",
            "s3.endpoint": S3_ENDPOINT,
            "s3.path-style-access": "true",
            "s3.region": S3_REGION,
            "s3.access-key-id": S3_ACCESS_KEY,
            "s3.secret-access-key": S3_SECRET_KEY
        },
        "storageConfigInfo": {
            "storageType": "S3",
            "stsUnavailable": True,
            "pathStyleAccess": True,
            "endpoint": S3_ENDPOINT,
            "region": S3_REGION,
            "allowedLocations": [
                f"s3://{S3_BUCKET}/"
            ]
        }
    }
}

r = requests.post(f"{POLARIS_URL}/api/management/v1/catalogs", headers=headers, json=catalog_payload)

if r.status_code in (200, 201):
    print("✅ Catalog 'lakehouse' registered (SeaweedFS + path-style)")
elif r.status_code == 409:
    print("⚠️  Catalog 'lakehouse' already exists — skipping")
else:
    print(f"❌ Catalog registration failed: {r.status_code} — {r.text}")

✅ Catalog 'lakehouse' registered (SeaweedFS + path-style)


---
## 🏷️ Step 6 — Create Medallion Namespaces

In [5]:
for ns in NAMESPACES:
    payload = {
        "namespace": [ns],
        "properties": {}
    }
    r = requests.post(
        f"{POLARIS_URL}/api/catalog/v1/lakehouse/namespaces",
        headers=headers,
        json=payload,
    )
    if r.status_code in (200, 201):
        print(f"✅ Namespace '{ns}' created → s3://{S3_BUCKET}/{ns}/")
    elif r.status_code == 409:
        print(f"⚠️  Namespace '{ns}' already exists — skipping")
    else:
        print(f"❌ Namespace '{ns}' failed: {r.status_code} — {r.text}")

print("\n🎉 Setup complete!")

✅ Namespace 'bronze' created → s3://lakehouse/bronze/
✅ Namespace 'silver' created → s3://lakehouse/silver/
✅ Namespace 'gold' created → s3://lakehouse/gold/

🎉 Setup complete!


---
# 🚀 Hello World — Query with Trino

Now let's connect to Trino and run some Iceberg queries.

## 🔌 Connect to Trino

In [6]:
from trino.dbapi import connect

conn = connect(
    host=TRINO_HOST,
    port=TRINO_PORT,
    user="admin",
    catalog="iceberg",
    schema="bronze",
)
cursor = conn.cursor()

def run_query(sql, display=True):
    """Execute a query and return results."""
    cursor.execute(sql)
    try:
        rows = cursor.fetchall()
        columns = [desc[0] for desc in cursor.description] if cursor.description else []
        if display and rows:
            widths = [max(len(str(c)), max(len(str(r[i])) for r in rows)) for i, c in enumerate(columns)]
            header = " | ".join(c.ljust(w) for c, w in zip(columns, widths))
            sep = "-+-".join("-" * w for w in widths)
            print(header)
            print(sep)
            for row in rows:
                print(" | ".join(str(v).ljust(w) for v, w in zip(row, widths)))
        return rows
    except Exception:
        return []

print("✅ Connected to Trino")

✅ Connected to Trino


## 📋 Show Schemas (Medallion Layers)

In [7]:
run_query("SHOW SCHEMAS FROM iceberg")

Schema            
------------------
bronze            
gold              
information_schema
silver            
system            


[['bronze'], ['gold'], ['information_schema'], ['silver'], ['system']]

## 🛠️ Create an Iceberg Table

In [8]:
run_query("""
CREATE TABLE IF NOT EXISTS iceberg.bronze.raw_events (
    event_id    BIGINT,
    event_type  VARCHAR,
    user_id     VARCHAR,
    payload     VARCHAR,
    event_time  TIMESTAMP(6) WITH TIME ZONE
)
WITH (
    format = 'PARQUET'
)
""")
print("✅ Table 'raw_events' created in bronze layer")

✅ Table 'raw_events' created in bronze layer


## 📥 Insert Sample Data

In [9]:
run_query("""
INSERT INTO iceberg.bronze.raw_events VALUES
    (1, 'page_view',  'user-001', '{"page": "/home"}',      TIMESTAMP '2025-01-15 10:30:00.000000 UTC'),
    (2, 'click',      'user-002', '{"button": "sign_up"}',   TIMESTAMP '2025-01-15 10:31:00.000000 UTC'),
    (3, 'purchase',   'user-001', '{"item": "pro_plan"}',    TIMESTAMP '2025-01-15 10:35:00.000000 UTC'),
    (4, 'page_view',  'user-003', '{"page": "/pricing"}',    TIMESTAMP '2025-01-15 10:40:00.000000 UTC'),
    (5, 'click',      'user-003', '{"button": "buy_now"}',   TIMESTAMP '2025-01-15 10:41:00.000000 UTC')
""")
print("✅ 5 events inserted")

rows
----
5   
✅ 5 events inserted


## 🔍 Query the Data

In [10]:
run_query("SELECT * FROM iceberg.bronze.raw_events ORDER BY event_time")

event_id | event_type | user_id  | payload               | event_time               
---------+------------+----------+-----------------------+--------------------------
1        | page_view  | user-001 | {"page": "/home"}     | 2025-01-15 10:30:00+00:00
2        | click      | user-002 | {"button": "sign_up"} | 2025-01-15 10:31:00+00:00
3        | purchase   | user-001 | {"item": "pro_plan"}  | 2025-01-15 10:35:00+00:00
4        | page_view  | user-003 | {"page": "/pricing"}  | 2025-01-15 10:40:00+00:00
5        | click      | user-003 | {"button": "buy_now"} | 2025-01-15 10:41:00+00:00


[[1,
  'page_view',
  'user-001',
  '{"page": "/home"}',
  datetime.datetime(2025, 1, 15, 10, 30, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [2,
  'click',
  'user-002',
  '{"button": "sign_up"}',
  datetime.datetime(2025, 1, 15, 10, 31, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [3,
  'purchase',
  'user-001',
  '{"item": "pro_plan"}',
  datetime.datetime(2025, 1, 15, 10, 35, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [4,
  'page_view',
  'user-003',
  '{"page": "/pricing"}',
  datetime.datetime(2025, 1, 15, 10, 40, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 [5,
  'click',
  'user-003',
  '{"button": "buy_now"}',
  datetime.datetime(2025, 1, 15, 10, 41, tzinfo=zoneinfo.ZoneInfo(key='UTC'))]]

## 📊 Analytical Query — Events per User

In [11]:
run_query("""
SELECT
    user_id,
    COUNT(*) AS total_events,
    COUNT(DISTINCT event_type) AS unique_event_types,
    MIN(event_time) AS first_event,
    MAX(event_time) AS last_event
FROM iceberg.bronze.raw_events
GROUP BY user_id
ORDER BY total_events DESC
""")

user_id  | total_events | unique_event_types | first_event               | last_event               
---------+--------------+--------------------+---------------------------+--------------------------
user-001 | 2            | 2                  | 2025-01-15 10:30:00+00:00 | 2025-01-15 10:35:00+00:00
user-003 | 2            | 2                  | 2025-01-15 10:40:00+00:00 | 2025-01-15 10:41:00+00:00
user-002 | 1            | 1                  | 2025-01-15 10:31:00+00:00 | 2025-01-15 10:31:00+00:00


[['user-001',
  2,
  2,
  datetime.datetime(2025, 1, 15, 10, 30, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  datetime.datetime(2025, 1, 15, 10, 35, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['user-003',
  2,
  2,
  datetime.datetime(2025, 1, 15, 10, 40, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  datetime.datetime(2025, 1, 15, 10, 41, tzinfo=zoneinfo.ZoneInfo(key='UTC'))],
 ['user-002',
  1,
  1,
  datetime.datetime(2025, 1, 15, 10, 31, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  datetime.datetime(2025, 1, 15, 10, 31, tzinfo=zoneinfo.ZoneInfo(key='UTC'))]]

## 🧊 Inspect Iceberg Metadata

In [12]:
run_query('SELECT * FROM iceberg.bronze."raw_events$snapshots"')

committed_at                     | snapshot_id         | parent_id           | operation | manifest_list                                                                                                                                   | summary                                                                                                                                                                                                                                                                                                                                                                                                                                                                                               
---------------------------------+---------------------+---------------------+-----------+-------------------------------------------------------------------------------------------------------------------------------------------------+----------------------------------------

[[datetime.datetime(2026, 2, 19, 14, 20, 54, 132000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  7436301542791440266,
  None,
  'append',
  's3://lakehouse/bronze/raw_events-a805806b60a942a1bdd6de8d15d1cc19/metadata/snap-7436301542791440266-1-38c54bad-d090-4b71-a1c5-bc53b0268310.avro',
  {'trino_query_id': '20260219_142053_00001_rnxdq',
   'trino_user': 'admin',
   'changed-partition-count': '0',
   'total-records': '0',
   'total-files-size': '0',
   'total-data-files': '0',
   'total-delete-files': '0',
   'total-position-deletes': '0',
   'total-equality-deletes': '0',
   'engine-version': '479',
   'engine-name': 'trino',
   'iceberg-version': 'Apache Iceberg 1.10.0 (commit 2114bf631e49af532d66e2ce148ee49dd1dd1f1f)'}],
 [datetime.datetime(2026, 2, 19, 14, 21, 12, 342000, tzinfo=zoneinfo.ZoneInfo(key='UTC')),
  4889881836371616966,
  7436301542791440266,
  'append',
  's3://lakehouse/bronze/raw_events-a805806b60a942a1bdd6de8d15d1cc19/metadata/snap-4889881836371616966-1-1d5edd83-433e-4d8

---
## 🧹 Cleanup (Optional)

Drop the table if you want to start fresh.

In [ ]:
# Uncomment to drop the table:
# run_query("DROP TABLE IF EXISTS iceberg.bronze.raw_events")
# print("🗑️ Table dropped")